# 13.6: Why the output distribution is exact

The first term equals $\min(q(x), p(x))$. The total rejection probability is $1 - \sum_{x'} \min(q(x'), p(x')) = \sum_{x'} \max(0, p(x') - q(x'))$, which is exactly the normalizer of $p'$, so the second term equals $\max(0, p(x) - q(x))$. Adding them gives $\min(q(x), p(x)) + \max(0, p(x) - q(x)) = p(x)$. The emitted token has exactly the target distribution, however poor the draft is. A bad draft only costs speed, not quality. (With greedy decoding, the rule reduces to accepting draft tokens as long as they match the target's argmax.)

Here is an implementation of the verification step, with a simulation confirming that the first emitted token follows the target distribution:


In [ ]:
import numpy as np

def speculative_step(p_target, q_draft, draft_tokens, rng):
    """One verification step of speculative sampling.
    q_draft[i]:  draft distribution used to sample draft_tokens[i]
    p_target[i]: target distribution at the same position (len(draft_tokens) + 1 rows,
                 the last row is for the position after all draft tokens)."""
    out = []
    for i, x in enumerate(draft_tokens):
        p, q = p_target[i], q_draft[i]
        if rng.random() < min(1.0, p[x] / q[x]):     # accept with prob min(1, p/q)
            out.append(x)
            continue
        residual = np.maximum(p - q, 0.0)            # reject: resample from (p - q)+
        out.append(int(rng.choice(len(p), p=residual / residual.sum())))
        return out
    out.append(int(rng.choice(len(p_target[-1]), p=p_target[-1])))   # all accepted: bonus token
    return out

# Check that the first emitted token follows the target distribution exactly.
rng = np.random.default_rng(0)
p = np.array([0.5, 0.3, 0.15, 0.05])
q = np.array([0.25, 0.25, 0.25, 0.25])
counts = np.zeros(4)
for _ in range(200_000):
    x = int(rng.choice(4, p=q))
    first = speculative_step([p, p], [q], [x], rng)[0]
    counts[first] += 1
print(np.round(counts / counts.sum(), 3))           # close to [0.5, 0.3, 0.15, 0.05]
